[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/Optimization-for-Machine-Learning/blob/main/notebooks/ch05_regularization.ipynb)

# Chapter 5: Regularization Techniques

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** why regularization is essential for generalization
2. **Implement** L1 (Lasso) and L2 (Ridge) regularization from scratch
3. **Visualize** how different regularization techniques affect weight distributions
4. **Build** and understand Dropout as a regularization technique
5. **Implement** early stopping to prevent overfitting
6. **Analyze** the effect of regularization strength on model performance

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# For reproducibility
np.random.seed(42)

print("Setup complete!")

## 1. The Bias-Variance Tradeoff and Overfitting

Before diving into regularization techniques, let's understand the fundamental problem they solve: **overfitting**.

### The Bias-Variance Decomposition

The expected prediction error can be decomposed as:

$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

- **High Bias**: Model is too simple, underfits the data
- **High Variance**: Model is too complex, overfits the training data

Regularization helps reduce variance at the cost of slightly increased bias.

In [ ]:
# Demonstrate overfitting with polynomial regression

def generate_polynomial_data(n_samples=30, noise=0.3, random_state=42):
    """Generate data from a simple polynomial with noise."""
    np.random.seed(random_state)
    X = np.sort(np.random.uniform(-1, 1, n_samples))
    # True function: y = 0.5 + 2*x - x^2
    y_true = 0.5 + 2*X - X**2
    y = y_true + np.random.normal(0, noise, n_samples)
    return X, y, y_true

def polynomial_features(X, degree):
    """Create polynomial features up to given degree."""
    return np.column_stack([X**i for i in range(degree + 1)])

def fit_polynomial(X, y, degree):
    """Fit polynomial regression using normal equations."""
    X_poly = polynomial_features(X, degree)
    # Normal equation: w = (X^T X)^{-1} X^T y
    w = np.linalg.lstsq(X_poly, y, rcond=None)[0]
    return w

def predict_polynomial(X, w):
    """Predict using polynomial weights."""
    degree = len(w) - 1
    X_poly = polynomial_features(X, degree)
    return X_poly @ w

# Generate data
X, y, y_true = generate_polynomial_data(n_samples=20, noise=0.3)

# Fit polynomials of different degrees
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
X_plot = np.linspace(-1.1, 1.1, 200)
degrees = [1, 2, 5, 10, 15, 20]

for ax, degree in zip(axes.flat, degrees):
    # Fit and predict
    w = fit_polynomial(X, y, degree)
    y_pred = predict_polynomial(X_plot, w)
    
    # Calculate training error
    y_train_pred = predict_polynomial(X, w)
    train_mse = np.mean((y - y_train_pred)**2)
    
    # Plot
    ax.scatter(X, y, c='blue', s=50, alpha=0.7, label='Training data')
    ax.plot(X_plot, y_pred, 'r-', linewidth=2, label=f'Degree {degree}')
    ax.plot(X_plot, 0.5 + 2*X_plot - X_plot**2, 'g--', linewidth=2, 
            alpha=0.7, label='True function')
    
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1, 3)
    ax.set_title(f'Degree {degree}\nTrain MSE: {train_mse:.4f}', fontsize=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Overfitting Demonstration: Polynomial Regression', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\nObservation: Higher degree polynomials fit training data perfectly")
print("but they overfit - notice the wild oscillations for degree 15 and 20.")

## 2. L1 vs L2 Regularization

### Mathematical Formulation

**L2 Regularization (Ridge)**:
$$\mathcal{L}_{\text{Ridge}} = \mathcal{L}_{\text{data}} + \lambda \sum_i w_i^2 = \mathcal{L}_{\text{data}} + \lambda ||\mathbf{w}||_2^2$$

**L1 Regularization (Lasso)**:
$$\mathcal{L}_{\text{Lasso}} = \mathcal{L}_{\text{data}} + \lambda \sum_i |w_i| = \mathcal{L}_{\text{data}} + \lambda ||\mathbf{w}||_1$$

### Key Differences

| Property | L1 (Lasso) | L2 (Ridge) |
|----------|------------|------------|
| Penalty shape | Diamond | Circle |
| Sparsity | Encourages sparse weights | Does not encourage sparsity |
| Feature selection | Yes (some weights become exactly 0) | No (weights shrink but rarely reach 0) |
| Solution | Unique (if features not collinear) | Always unique |
| Computational | May need special solvers | Closed-form solution exists |

In [ ]:
# Visualize L1 vs L2 constraint regions

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a grid for contour plot
w1 = np.linspace(-3, 3, 200)
w2 = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1, w2)

# Define a simple loss function: (w1 - 2)^2 + (w2 - 2)^2
# This represents the contours of the unregularized loss
loss = (W1 - 2)**2 + (W2 - 2)**2

# Plot 1: L1 constraint region
ax = axes[0]
ax.contour(W1, W2, loss, levels=20, cmap='RdYlBu_r', alpha=0.7)

# L1 ball: |w1| + |w2| <= 1
diamond = Polygon([(-1, 0), (0, 1), (1, 0), (0, -1)], 
                  fill=True, facecolor='lightblue', edgecolor='blue', 
                  linewidth=2, alpha=0.5)
ax.add_patch(diamond)

# Mark the constrained optimum (touches corner -> sparse solution)
ax.plot(0, 1, 'r*', markersize=15, label='L1 solution (sparse)')
ax.plot(2, 2, 'go', markersize=10, label='Unconstrained optimum')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('$w_1$', fontsize=14)
ax.set_ylabel('$w_2$', fontsize=14)
ax.set_title('L1 Regularization (Lasso)\nDiamond constraint region', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax.set_aspect('equal')

# Plot 2: L2 constraint region
ax = axes[1]
ax.contour(W1, W2, loss, levels=20, cmap='RdYlBu_r', alpha=0.7)

# L2 ball: w1^2 + w2^2 <= 1
circle = Circle((0, 0), 1, fill=True, facecolor='lightgreen', 
                edgecolor='green', linewidth=2, alpha=0.5)
ax.add_patch(circle)

# L2 solution (tangent to circle, not at corner)
# The solution lies along the line from (0,0) to (2,2)
l2_sol = np.array([2, 2]) / np.linalg.norm([2, 2])
ax.plot(*l2_sol, 'r*', markersize=15, label='L2 solution (dense)')
ax.plot(2, 2, 'go', markersize=10, label='Unconstrained optimum')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('$w_1$', fontsize=14)
ax.set_ylabel('$w_2$', fontsize=14)
ax.set_title('L2 Regularization (Ridge)\nCircular constraint region', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax.set_aspect('equal')

# Plot 3: Comparison of penalty functions
ax = axes[2]
w = np.linspace(-2, 2, 200)
ax.plot(w, np.abs(w), 'b-', linewidth=2, label='L1: $|w|$')
ax.plot(w, w**2, 'g-', linewidth=2, label='L2: $w^2$')
ax.plot(w, np.abs(w)**0.5, 'r--', linewidth=2, label='L0.5: $|w|^{0.5}$')

ax.set_xlabel('$w$', fontsize=14)
ax.set_ylabel('Penalty', fontsize=14)
ax.set_title('Penalty Functions', fontsize=12)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Implement L1 and L2 regularized linear regression from scratch

class LinearRegressionL2:
    """Linear regression with L2 (Ridge) regularization.
    
    Closed-form solution: w = (X^T X + lambda * I)^{-1} X^T y
    """
    
    def __init__(self, lambda_reg=1.0):
        self.lambda_reg = lambda_reg
        self.w = None
    
    def fit(self, X, y):
        """Fit the model using closed-form solution."""
        n_features = X.shape[1]
        # (X^T X + lambda * I)^{-1} X^T y
        XtX = X.T @ X
        regularization = self.lambda_reg * np.eye(n_features)
        self.w = np.linalg.solve(XtX + regularization, X.T @ y)
        return self
    
    def predict(self, X):
        """Predict using the fitted model."""
        return X @ self.w
    
    def get_weights(self):
        return self.w


class LinearRegressionL1:
    """Linear regression with L1 (Lasso) regularization.
    
    Uses coordinate descent for optimization since there's no 
    closed-form solution.
    """
    
    def __init__(self, lambda_reg=1.0, max_iter=1000, tol=1e-6):
        self.lambda_reg = lambda_reg
        self.max_iter = max_iter
        self.tol = tol
        self.w = None
        self.loss_history = []
    
    def soft_threshold(self, x, threshold):
        """Soft thresholding operator for L1 proximal update."""
        return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)
    
    def fit(self, X, y):
        """Fit using coordinate descent."""
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.loss_history = []
        
        # Precompute X^T X diagonal and X^T y
        X_sq_sum = np.sum(X**2, axis=0)
        
        for iteration in range(self.max_iter):
            w_old = self.w.copy()
            
            # Coordinate descent: update each weight one at a time
            for j in range(n_features):
                # Compute residual without feature j
                residual = y - X @ self.w + X[:, j] * self.w[j]
                
                # Compute the optimal update for coordinate j
                rho_j = X[:, j] @ residual
                
                # Apply soft thresholding
                if X_sq_sum[j] > 0:
                    self.w[j] = self.soft_threshold(
                        rho_j / X_sq_sum[j], 
                        self.lambda_reg / X_sq_sum[j]
                    )
            
            # Check convergence
            if np.max(np.abs(self.w - w_old)) < self.tol:
                break
            
            # Record loss
            loss = 0.5 * np.mean((y - X @ self.w)**2) + self.lambda_reg * np.sum(np.abs(self.w))
            self.loss_history.append(loss)
        
        return self
    
    def predict(self, X):
        """Predict using the fitted model."""
        return X @ self.w
    
    def get_weights(self):
        return self.w

print("L1 and L2 regularized regression classes defined!")

In [ ]:
# Compare L1 vs L2 on a regression task with many features

# Generate data with sparse true weights
np.random.seed(42)
n_samples = 200
n_features = 50

# True weights: only 5 features are actually relevant
w_true = np.zeros(n_features)
w_true[:5] = [3, -2, 1.5, -1, 0.5]  # Only first 5 features matter

# Generate features and target
X = np.random.randn(n_samples, n_features)
y = X @ w_true + np.random.randn(n_samples) * 0.5

# Fit with different regularization strengths
lambdas = [0.001, 0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(2, len(lambdas), figsize=(18, 8))

for i, lam in enumerate(lambdas):
    # L2 (Ridge)
    model_l2 = LinearRegressionL2(lambda_reg=lam)
    model_l2.fit(X, y)
    w_l2 = model_l2.get_weights()
    
    # L1 (Lasso)
    model_l1 = LinearRegressionL1(lambda_reg=lam, max_iter=2000)
    model_l1.fit(X, y)
    w_l1 = model_l1.get_weights()
    
    # Plot L2 weights
    ax = axes[0, i]
    ax.bar(range(n_features), w_l2, alpha=0.7, color='green', label='L2 weights')
    ax.bar(range(n_features), w_true, alpha=0.3, color='blue', label='True weights')
    ax.set_title(f'L2 (Ridge), $\\lambda$={lam}', fontsize=11)
    ax.set_xlabel('Feature index')
    if i == 0:
        ax.set_ylabel('Weight value')
        ax.legend(fontsize=8)
    ax.set_ylim(-4, 4)
    
    # Plot L1 weights
    ax = axes[1, i]
    ax.bar(range(n_features), w_l1, alpha=0.7, color='red', label='L1 weights')
    ax.bar(range(n_features), w_true, alpha=0.3, color='blue', label='True weights')
    ax.set_title(f'L1 (Lasso), $\\lambda$={lam}\nNon-zero: {np.sum(np.abs(w_l1) > 0.01)}', fontsize=11)
    ax.set_xlabel('Feature index')
    if i == 0:
        ax.set_ylabel('Weight value')
        ax.legend(fontsize=8)
    ax.set_ylim(-4, 4)

plt.suptitle('L1 vs L2 Regularization: Weight Comparison\n(True model has only 5 non-zero weights)', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Observation:")
print("- L2 (Ridge) shrinks all weights but keeps them non-zero")
print("- L1 (Lasso) produces sparse solutions, setting many weights exactly to zero")

In [ ]:
# Visualize weight distributions

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Different regularization strengths
lambdas_to_compare = [0.01, 0.1, 1.0]

for ax, lam in zip(axes, lambdas_to_compare):
    # L2
    model_l2 = LinearRegressionL2(lambda_reg=lam)
    model_l2.fit(X, y)
    w_l2 = model_l2.get_weights()
    
    # L1
    model_l1 = LinearRegressionL1(lambda_reg=lam, max_iter=2000)
    model_l1.fit(X, y)
    w_l1 = model_l1.get_weights()
    
    # Histogram of weight values
    bins = np.linspace(-3, 3, 30)
    ax.hist(w_l2, bins=bins, alpha=0.5, color='green', label='L2 (Ridge)', density=True)
    ax.hist(w_l1, bins=bins, alpha=0.5, color='red', label='L1 (Lasso)', density=True)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Weight value', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'$\\lambda$ = {lam}\nL1 zeros: {np.sum(np.abs(w_l1) < 0.01)}/{n_features}', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Weight Distribution: L1 vs L2 Regularization', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Dropout Implementation and Visualization

Dropout is a powerful regularization technique for neural networks that randomly "drops" (sets to zero) a fraction of neurons during training.

### How Dropout Works

**During Training:**
$$h = f(W \cdot x) \odot m$$
where $m \sim \text{Bernoulli}(1 - p)$ and $\odot$ is element-wise multiplication.

**During Inference:**
$$h = (1 - p) \cdot f(W \cdot x)$$
or equivalently, scale during training (inverted dropout).

### Why Dropout Works

1. **Prevents co-adaptation**: Forces neurons to learn independently useful features
2. **Implicit ensemble**: Acts like training many different networks and averaging
3. **Weight regularization**: Approximately equivalent to L2 regularization

In [ ]:
class Dropout:
    """Dropout layer implementation (inverted dropout)."""
    
    def __init__(self, p=0.5):
        """Initialize dropout layer.
        
        Parameters:
        -----------
        p : float
            Dropout probability (probability of dropping a unit)
        """
        self.p = p
        self.mask = None
        self.training = True
    
    def forward(self, x):
        """Forward pass through dropout layer."""
        if self.training and self.p > 0:
            # Create binary mask
            self.mask = (np.random.rand(*x.shape) > self.p).astype(float)
            # Scale by 1/(1-p) to maintain expected value (inverted dropout)
            return x * self.mask / (1 - self.p)
        else:
            return x
    
    def backward(self, grad_output):
        """Backward pass - gradient flows only through non-dropped units."""
        if self.training and self.p > 0:
            return grad_output * self.mask / (1 - self.p)
        else:
            return grad_output
    
    def train(self):
        self.training = True
    
    def eval(self):
        self.training = False


# Visualize dropout in action
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Create a simple input
np.random.seed(42)
x = np.ones((8, 8))  # 8x8 grid

dropout_rates = [0.0, 0.25, 0.5, 0.75]

for i, p in enumerate(dropout_rates):
    dropout = Dropout(p=p)
    dropout.train()
    
    # Show the mask
    _ = dropout.forward(x)  # Generate mask
    mask = dropout.mask if dropout.mask is not None else np.ones_like(x)
    
    ax = axes[0, i]
    im = ax.imshow(mask, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f'Dropout Mask (p={p})\nDropped: {np.sum(mask == 0)}/{mask.size}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Show the output (scaled)
    output = dropout.forward(x)
    ax = axes[1, i]
    im = ax.imshow(output, cmap='viridis', vmin=0, vmax=2)
    ax.set_title(f'Output (scaled)\nExpected value: {output.mean():.2f}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Dropout Visualization: Masks and Outputs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Despite dropout, the expected output value remains ~1.0")
print("due to the scaling factor 1/(1-p)")

In [ ]:
# Implement a simple neural network with dropout

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_loss(y_pred, y_true):
    n_samples = y_true.shape[0]
    # Clip to prevent log(0)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.sum(y_true * np.log(y_pred)) / n_samples


class SimpleNeuralNetwork:
    """Simple 2-layer neural network with optional dropout."""
    
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.0):
        # Initialize weights with Xavier initialization
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(output_dim)
        
        self.dropout = Dropout(p=dropout_rate)
        self.dropout_rate = dropout_rate
        
        # For storing intermediate values
        self.cache = {}
    
    def forward(self, X, training=True):
        """Forward pass."""
        if training:
            self.dropout.train()
        else:
            self.dropout.eval()
        
        # Layer 1
        self.cache['z1'] = X @ self.W1 + self.b1
        self.cache['a1'] = relu(self.cache['z1'])
        
        # Apply dropout after activation
        self.cache['a1_dropout'] = self.dropout.forward(self.cache['a1'])
        
        # Layer 2
        self.cache['z2'] = self.cache['a1_dropout'] @ self.W2 + self.b2
        self.cache['a2'] = softmax(self.cache['z2'])
        
        return self.cache['a2']
    
    def backward(self, X, y_true, learning_rate=0.01):
        """Backward pass with gradient descent update."""
        n_samples = X.shape[0]
        
        # Output layer gradient
        dz2 = self.cache['a2'] - y_true
        dW2 = self.cache['a1_dropout'].T @ dz2 / n_samples
        db2 = np.mean(dz2, axis=0)
        
        # Backpropagate through dropout
        da1 = dz2 @ self.W2.T
        da1 = self.dropout.backward(da1)
        
        # Hidden layer gradient
        dz1 = da1 * relu_derivative(self.cache['z1'])
        dW1 = X.T @ dz1 / n_samples
        db1 = np.mean(dz1, axis=0)
        
        # Update weights
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
    
    def train_epoch(self, X, y, learning_rate=0.01):
        """Train for one epoch."""
        y_pred = self.forward(X, training=True)
        loss = cross_entropy_loss(y_pred, y)
        self.backward(X, y, learning_rate)
        return loss
    
    def evaluate(self, X, y):
        """Evaluate model (no dropout during inference)."""
        y_pred = self.forward(X, training=False)
        loss = cross_entropy_loss(y_pred, y)
        accuracy = np.mean(np.argmax(y_pred, axis=1) == np.argmax(y, axis=1))
        return loss, accuracy

print("Neural network with dropout defined!")

In [ ]:
# Train networks with and without dropout and compare

# Generate classification data
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                          n_redundant=5, n_classes=3, n_clusters_per_class=1,
                          random_state=42)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# One-hot encode targets
y_onehot = np.zeros((y.shape[0], 3))
y_onehot[np.arange(y.shape[0]), y] = 1

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y_onehot, test_size=0.3, random_state=42)

# Train models with different dropout rates
dropout_rates = [0.0, 0.2, 0.5]
n_epochs = 200

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for dropout_rate, color in zip(dropout_rates, colors):
    np.random.seed(42)  # Reset for fair comparison
    model = SimpleNeuralNetwork(20, 64, 3, dropout_rate=dropout_rate)
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    for epoch in range(n_epochs):
        # Train
        train_loss = model.train_epoch(X_train, y_train, learning_rate=0.5)
        
        # Evaluate
        _, train_acc = model.evaluate(X_train, y_train)
        val_loss, val_acc = model.evaluate(X_val, y_val)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
    
    # Plot loss curves
    label = f'Dropout {dropout_rate}'
    axes[0].plot(train_losses, color=color, linestyle='-', linewidth=2, 
                 label=f'{label} (train)', alpha=0.8)
    axes[0].plot(val_losses, color=color, linestyle='--', linewidth=2, 
                 label=f'{label} (val)', alpha=0.8)
    
    # Plot accuracy curves
    axes[1].plot(train_accs, color=color, linestyle='-', linewidth=2, 
                 label=f'{label} (train)', alpha=0.8)
    axes[1].plot(val_accs, color=color, linestyle='--', linewidth=2, 
                 label=f'{label} (val)', alpha=0.8)
    
    print(f"Dropout {dropout_rate}: Final val acc = {val_accs[-1]:.4f}, "
          f"Train-val gap = {train_accs[-1] - val_accs[-1]:.4f}")

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14)
axes[0].legend(fontsize=9, loc='upper right')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14)
axes[1].legend(fontsize=9, loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Dropout on Training Dynamics', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Early Stopping with Validation Curves

Early stopping is a simple but effective regularization technique: stop training when validation performance stops improving.

### Algorithm

1. Monitor validation loss/accuracy at each epoch
2. Keep track of the best model seen so far
3. If validation performance doesn't improve for `patience` epochs, stop training
4. Return the best model

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting.
    
    Stops training when validation loss doesn't improve for `patience` epochs.
    """
    
    def __init__(self, patience=10, min_delta=1e-4, mode='min'):
        """Initialize early stopping.
        
        Parameters:
        -----------
        patience : int
            Number of epochs with no improvement after which training stops
        min_delta : float
            Minimum change to qualify as an improvement
        mode : str
            'min' for loss (lower is better), 'max' for accuracy (higher is better)
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
        self.best_model_weights = None
    
    def __call__(self, score, epoch, model_weights=None):
        """Check if training should stop.
        
        Parameters:
        -----------
        score : float
            Current validation score
        epoch : int
            Current epoch
        model_weights : dict
            Dictionary of model weights to save
        
        Returns:
        --------
        bool
            True if training should stop
        """
        if self.mode == 'min':
            is_better = self.best_score is None or score < self.best_score - self.min_delta
        else:  # mode == 'max'
            is_better = self.best_score is None or score > self.best_score + self.min_delta
        
        if is_better:
            self.best_score = score
            self.counter = 0
            self.best_epoch = epoch
            if model_weights is not None:
                self.best_model_weights = {k: v.copy() for k, v in model_weights.items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop
    
    def reset(self):
        """Reset early stopping state."""
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
        self.best_model_weights = None

print("Early stopping class defined!")

In [ ]:
# Demonstrate early stopping

np.random.seed(42)

# Use a deliberately overparameterized network to induce overfitting
model = SimpleNeuralNetwork(20, 128, 3, dropout_rate=0.0)
early_stopping = EarlyStopping(patience=15, min_delta=0.001, mode='min')

n_epochs = 300
train_losses = []
val_losses = []
train_accs = []
val_accs = []

stopped_epoch = n_epochs

for epoch in range(n_epochs):
    # Train
    train_loss = model.train_epoch(X_train, y_train, learning_rate=0.5)
    
    # Evaluate
    _, train_acc = model.evaluate(X_train, y_train)
    val_loss, val_acc = model.evaluate(X_val, y_val)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Check early stopping
    model_weights = {'W1': model.W1, 'b1': model.b1, 'W2': model.W2, 'b2': model.b2}
    if early_stopping(val_loss, epoch, model_weights):
        stopped_epoch = epoch
        print(f"\nEarly stopping triggered at epoch {epoch}")
        print(f"Best epoch: {early_stopping.best_epoch} with val loss: {early_stopping.best_score:.4f}")
        break

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
ax.plot(train_losses, 'b-', linewidth=2, label='Training loss', alpha=0.8)
ax.plot(val_losses, 'r-', linewidth=2, label='Validation loss', alpha=0.8)
ax.axvline(x=early_stopping.best_epoch, color='green', linestyle='--', linewidth=2,
           label=f'Best epoch ({early_stopping.best_epoch})')
if stopped_epoch < n_epochs:
    ax.axvline(x=stopped_epoch, color='orange', linestyle=':', linewidth=2,
               label=f'Stopped epoch ({stopped_epoch})')
ax.fill_between(range(early_stopping.best_epoch, len(val_losses)), 
                min(val_losses), max(train_losses),
                alpha=0.2, color='red', label='Overfitting region')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training and Validation Loss', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Accuracy curves
ax = axes[1]
ax.plot(train_accs, 'b-', linewidth=2, label='Training accuracy', alpha=0.8)
ax.plot(val_accs, 'r-', linewidth=2, label='Validation accuracy', alpha=0.8)
ax.axvline(x=early_stopping.best_epoch, color='green', linestyle='--', linewidth=2,
           label=f'Best epoch ({early_stopping.best_epoch})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training and Validation Accuracy', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Early Stopping Demonstration', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nWithout early stopping: Final val acc = {val_accs[-1]:.4f}")
print(f"With early stopping: Best val acc = {val_accs[early_stopping.best_epoch]:.4f}")

## 5. Overfitting Live Demo: Train vs Val Divergence

Let's create a clear demonstration of overfitting where we can see the train and validation curves diverge in real-time.

In [ ]:
# Create a scenario that's prone to overfitting:
# Small dataset + complex model

np.random.seed(42)

# Generate a small dataset
X_small, y_small = make_classification(
    n_samples=100,  # Small dataset
    n_features=50,  # Many features
    n_informative=5,  # But only 5 are useful
    n_redundant=10,
    n_classes=2,
    random_state=42
)

# Standardize
scaler = StandardScaler()
X_small = scaler.fit_transform(X_small)

# One-hot encode
y_small_onehot = np.zeros((y_small.shape[0], 2))
y_small_onehot[np.arange(y_small.shape[0]), y_small] = 1

# Very small validation set (realistic for limited data scenarios)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_small, y_small_onehot, test_size=0.3, random_state=42
)

print(f"Training samples: {X_train_s.shape[0]}")
print(f"Validation samples: {X_val_s.shape[0]}")
print(f"Features: {X_train_s.shape[1]}")

In [ ]:
# Train with different model complexities and regularization

configurations = [
    {'name': 'Small model (16 hidden)', 'hidden_dim': 16, 'dropout': 0.0},
    {'name': 'Large model (256 hidden)', 'hidden_dim': 256, 'dropout': 0.0},
    {'name': 'Large model + Dropout 0.5', 'hidden_dim': 256, 'dropout': 0.5},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
n_epochs = 500

for ax, config in zip(axes, configurations):
    np.random.seed(42)
    model = SimpleNeuralNetwork(
        50, config['hidden_dim'], 2, 
        dropout_rate=config['dropout']
    )
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    for epoch in range(n_epochs):
        train_loss = model.train_epoch(X_train_s, y_train_s, learning_rate=0.1)
        _, train_acc = model.evaluate(X_train_s, y_train_s)
        val_loss, val_acc = model.evaluate(X_val_s, y_val_s)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
    
    # Plot
    ax.plot(train_accs, 'b-', linewidth=2, label='Train', alpha=0.8)
    ax.plot(val_accs, 'r-', linewidth=2, label='Validation', alpha=0.8)
    
    # Mark overfitting region
    best_val_epoch = np.argmax(val_accs)
    ax.axvline(x=best_val_epoch, color='green', linestyle='--', 
               label=f'Best val @ {best_val_epoch}')
    
    gap = train_accs[-1] - val_accs[-1]
    ax.set_title(f"{config['name']}\nFinal gap: {gap:.3f}", fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.4, 1.05)

plt.suptitle('Overfitting Analysis: Model Complexity and Regularization', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Regularization Strength Sweep

Let's systematically analyze how different regularization strengths affect model performance.

In [ ]:
# Regularization strength sweep for L2

# Generate data with known structure
np.random.seed(42)
n_samples = 200
n_features = 30
n_informative = 5

# True weights
w_true = np.zeros(n_features)
w_true[:n_informative] = np.array([3, -2, 1.5, -1, 0.5])

# Generate data
X = np.random.randn(n_samples, n_features)
y = X @ w_true + np.random.randn(n_samples) * 0.5

# Split
X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(X, y, test_size=0.3, random_state=42)

# Test different regularization strengths
lambdas = np.logspace(-4, 2, 50)

train_mses_l2 = []
val_mses_l2 = []
train_mses_l1 = []
val_mses_l1 = []
n_nonzero_l1 = []

for lam in lambdas:
    # L2
    model_l2 = LinearRegressionL2(lambda_reg=lam)
    model_l2.fit(X_train_r, y_train_r)
    train_pred_l2 = model_l2.predict(X_train_r)
    val_pred_l2 = model_l2.predict(X_val_r)
    train_mses_l2.append(np.mean((y_train_r - train_pred_l2)**2))
    val_mses_l2.append(np.mean((y_val_r - val_pred_l2)**2))
    
    # L1
    model_l1 = LinearRegressionL1(lambda_reg=lam, max_iter=2000)
    model_l1.fit(X_train_r, y_train_r)
    train_pred_l1 = model_l1.predict(X_train_r)
    val_pred_l1 = model_l1.predict(X_val_r)
    train_mses_l1.append(np.mean((y_train_r - train_pred_l1)**2))
    val_mses_l1.append(np.mean((y_val_r - val_pred_l1)**2))
    n_nonzero_l1.append(np.sum(np.abs(model_l1.get_weights()) > 0.01))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# L2 regularization sweep
ax = axes[0]
ax.semilogx(lambdas, train_mses_l2, 'b-', linewidth=2, label='Train MSE')
ax.semilogx(lambdas, val_mses_l2, 'r-', linewidth=2, label='Validation MSE')
best_idx_l2 = np.argmin(val_mses_l2)
ax.axvline(x=lambdas[best_idx_l2], color='green', linestyle='--', 
           label=f'Best lambda: {lambdas[best_idx_l2]:.4f}')
ax.set_xlabel('Regularization strength ($\\lambda$)', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('L2 (Ridge) Regularization Sweep', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# L1 regularization sweep
ax = axes[1]
ax.semilogx(lambdas, train_mses_l1, 'b-', linewidth=2, label='Train MSE')
ax.semilogx(lambdas, val_mses_l1, 'r-', linewidth=2, label='Validation MSE')
best_idx_l1 = np.argmin(val_mses_l1)
ax.axvline(x=lambdas[best_idx_l1], color='green', linestyle='--',
           label=f'Best lambda: {lambdas[best_idx_l1]:.4f}')
ax.set_xlabel('Regularization strength ($\\lambda$)', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('L1 (Lasso) Regularization Sweep', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Sparsity vs lambda for L1
ax = axes[2]
ax.semilogx(lambdas, n_nonzero_l1, 'purple', linewidth=2)
ax.axhline(y=n_informative, color='green', linestyle='--', 
           label=f'True non-zero: {n_informative}')
ax.set_xlabel('Regularization strength ($\\lambda$)', fontsize=12)
ax.set_ylabel('Number of non-zero weights', fontsize=12)
ax.set_title('L1 Sparsity vs Regularization', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nL2 Best lambda: {lambdas[best_idx_l2]:.4f}, Val MSE: {val_mses_l2[best_idx_l2]:.4f}")
print(f"L1 Best lambda: {lambdas[best_idx_l1]:.4f}, Val MSE: {val_mses_l1[best_idx_l1]:.4f}, Non-zero: {n_nonzero_l1[best_idx_l1]}")

In [ ]:
# Dropout rate sweep

dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
final_train_accs = []
final_val_accs = []
best_val_accs = []

for dropout in dropout_rates:
    np.random.seed(42)
    model = SimpleNeuralNetwork(50, 128, 2, dropout_rate=dropout)
    
    train_accs = []
    val_accs = []
    
    for epoch in range(300):
        model.train_epoch(X_train_s, y_train_s, learning_rate=0.1)
        _, train_acc = model.evaluate(X_train_s, y_train_s)
        _, val_acc = model.evaluate(X_val_s, y_val_s)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
    
    final_train_accs.append(train_accs[-1])
    final_val_accs.append(val_accs[-1])
    best_val_accs.append(max(val_accs))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(dropout_rates, final_train_accs, 'b-o', linewidth=2, markersize=8, label='Final Train Acc')
ax.plot(dropout_rates, final_val_accs, 'r-o', linewidth=2, markersize=8, label='Final Val Acc')
ax.plot(dropout_rates, best_val_accs, 'g--s', linewidth=2, markersize=8, label='Best Val Acc')

# Mark the generalization gap
for i, (train, val) in enumerate(zip(final_train_accs, final_val_accs)):
    ax.fill_between([dropout_rates[i] - 0.02, dropout_rates[i] + 0.02], 
                    [val, val], [train, train], alpha=0.3, color='gray')

best_dropout_idx = np.argmax(best_val_accs)
ax.axvline(x=dropout_rates[best_dropout_idx], color='green', linestyle=':', 
           linewidth=2, label=f'Best dropout: {dropout_rates[best_dropout_idx]}')

ax.set_xlabel('Dropout Rate', fontsize=14)
ax.set_ylabel('Accuracy', fontsize=14)
ax.set_title('Effect of Dropout Rate on Model Performance', fontsize=16)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(dropout_rates)

plt.tight_layout()
plt.show()

print(f"\nBest dropout rate: {dropout_rates[best_dropout_idx]}")
print(f"Best validation accuracy: {best_val_accs[best_dropout_idx]:.4f}")

---

## Interview Questions

### Question 1: Explain the difference between L1 and L2 regularization. When would you use each?

**Answer:**

**L2 (Ridge) Regularization:**
- Adds $\lambda \sum w_i^2$ to the loss
- Shrinks weights towards zero but rarely exactly zero
- Has a closed-form solution
- Good when all features contribute somewhat to the prediction
- More stable with correlated features

**L1 (Lasso) Regularization:**
- Adds $\lambda \sum |w_i|$ to the loss
- Produces sparse solutions (many weights exactly zero)
- Effectively performs feature selection
- Good when you expect only a few features to be relevant
- Can be unstable with highly correlated features

**When to use:**
- Use L2 when you believe most features contribute and want to prevent large weights
- Use L1 when you want automatic feature selection or interpretability
- Use Elastic Net (L1 + L2) when you want the best of both worlds

### Question 2: Why does dropout work as a regularizer?

**Answer:**

Dropout works through several mechanisms:

1. **Prevents co-adaptation**: Neurons can't rely on specific other neurons being present, forcing them to learn more robust features independently

2. **Implicit ensemble**: Dropout is approximately equivalent to training many different networks (with shared weights) and averaging their predictions

3. **Noise injection**: Adding noise to hidden layers forces the network to learn representations that are robust to perturbations

4. **Weight regularization**: It can be shown that dropout is approximately equivalent to L2 regularization on the weights, scaled by the dropout probability

### Question 3: How do you choose the right regularization strength?

**Answer:**

The regularization strength is typically chosen through:

1. **Cross-validation**: The gold standard is to use k-fold cross-validation to find the regularization strength that gives the best validation performance

2. **Validation curve analysis**: Plot training and validation loss/accuracy for different regularization strengths:
   - Too low: Large gap between train and val (overfitting)
   - Too high: Both train and val performance suffer (underfitting)
   - Just right: Small gap with good overall performance

3. **Grid/Random search**: Search over a range of values (often logarithmically spaced)

4. **Bayesian optimization**: More sophisticated search methods for large hyperparameter spaces

5. **Rules of thumb**: For L2, start with 0.01-0.1; for dropout, start with 0.2-0.5

---

## Exercises

### Exercise 1: Implement Elastic Net Regularization

Elastic Net combines L1 and L2 regularization:

$$\mathcal{L} = \mathcal{L}_{\text{data}} + \lambda_1 ||\mathbf{w}||_1 + \lambda_2 ||\mathbf{w}||_2^2$$

In [ ]:
# Exercise 1: Implement Elastic Net

class ElasticNet:
    """Linear regression with Elastic Net regularization.
    
    TODO: Implement Elastic Net using coordinate descent
    Hint: Modify the L1 implementation to include an L2 penalty
    """
    
    def __init__(self, lambda_l1=1.0, lambda_l2=1.0, max_iter=1000, tol=1e-6):
        self.lambda_l1 = lambda_l1
        self.lambda_l2 = lambda_l2
        self.max_iter = max_iter
        self.tol = tol
        self.w = None
    
    def fit(self, X, y):
        # YOUR CODE HERE
        pass
    
    def predict(self, X):
        return X @ self.w
    
    def get_weights(self):
        return self.w

### Exercise 2: Implement Batch Normalization

Batch normalization is another form of regularization that normalizes layer inputs.

In [ ]:
# Exercise 2: Implement Batch Normalization

class BatchNorm:
    """Batch Normalization layer.
    
    TODO: Implement batch normalization
    During training: normalize using batch statistics
    During inference: use running averages
    """
    
    def __init__(self, num_features, momentum=0.1, epsilon=1e-5):
        self.num_features = num_features
        self.momentum = momentum
        self.epsilon = epsilon
        
        # Learnable parameters
        self.gamma = np.ones(num_features)   # Scale
        self.beta = np.zeros(num_features)   # Shift
        
        # Running statistics
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
        
        self.training = True
    
    def forward(self, x):
        # YOUR CODE HERE
        pass
    
    def backward(self, grad_output):
        # YOUR CODE HERE
        pass

### Exercise 3: Compare Regularization Techniques on Real Data

Load a real dataset (e.g., from sklearn) and compare:
1. No regularization
2. L1 regularization
3. L2 regularization
4. Dropout
5. Early stopping

Create a comprehensive comparison report.

In [ ]:
# Exercise 3: Comprehensive comparison

from sklearn.datasets import load_breast_cancer

# Load data
data = load_breast_cancer()
X_real = data.data
y_real = data.target

# YOUR CODE HERE:
# 1. Preprocess the data
# 2. Train models with each regularization technique
# 3. Compare and visualize results
# 4. Create a summary table

---

## Summary

In this notebook, we covered:

1. **L1 vs L2 Regularization**: L1 produces sparse solutions (feature selection), L2 shrinks weights uniformly

2. **Dropout**: Randomly drops neurons during training to prevent co-adaptation and act as implicit ensemble

3. **Early Stopping**: Monitor validation performance and stop when it stops improving

4. **Regularization Strength**: Must be tuned carefully - too low leads to overfitting, too high leads to underfitting

### Key Takeaways

- Regularization is essential for preventing overfitting
- Different techniques have different strengths:
  - L1 for feature selection
  - L2 for stable training
  - Dropout for neural networks
  - Early stopping as a simple, effective technique
- Always use validation data to tune regularization strength
- Combine multiple techniques for best results